### Monthly Sales Trend

In [0]:
%sql
CREATE OR REPLACE TABLE gold.monthly_trend
USING DELTA
AS
SELECT
  s.order_year,
  s.order_month,
 
  -- Clean label for charts — e.g. '2023-06'
  CONCAT(
    s.order_year, '-',
    LPAD(CAST(s.order_month AS STRING), 2, '0')
  )                                               AS year_month,
 
  COUNT(DISTINCT s.order_number)                  AS total_orders,
  SUM(s.quantity)                                 AS units_sold,
  ROUND(SUM(s.sales_amount), 2)                   AS total_revenue,
  ROUND(AVG(s.sales_amount), 2)                   AS avg_order_value,
 
  -- LAG looks at the previous row — last month's revenue
  ROUND(
    SUM(s.sales_amount) - LAG(SUM(s.sales_amount))
    OVER (ORDER BY s.order_year, s.order_month), 2
  )                                               AS revenue_change_vs_last_month,
 
  -- Month over month % change
  ROUND(
    (SUM(s.sales_amount) - LAG(SUM(s.sales_amount))
    OVER (ORDER BY s.order_year, s.order_month))
    / NULLIF(LAG(SUM(s.sales_amount))
    OVER (ORDER BY s.order_year, s.order_month), 0) * 100, 2
  )                                               AS revenue_change_pct,
 
  -- Running total — cumulative revenue from day one
  ROUND(
    SUM(SUM(s.sales_amount))
    OVER (ORDER BY s.order_year, s.order_month
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2
  )                                               AS cumulative_revenue
 
FROM workspace.silver.sales s
GROUP BY s.order_year, s.order_month
ORDER BY s.order_year, s.order_month;



Weekly Trend

In [0]:
%sql
CREATE OR REPLACE TABLE gold.weekly_trend
USING DELTA
AS
SELECT
  order_year,
  WEEKOFYEAR(order_date)                          AS order_week,
  COUNT(DISTINCT order_number)                    AS total_orders,
  SUM(quantity)                                   AS units_sold,
  ROUND(SUM(sales_amount), 2)                     AS total_revenue,
  ROUND(AVG(sales_amount), 2)                     AS avg_order_value
 
FROM workspace.silver.sales
GROUP BY order_year, WEEKOFYEAR(order_date)
ORDER BY order_year, order_week;
